# Inspección de Silver

## Configuración

In [ ]:
from pathlib import Path
import duckdb
from IPython.display import display, Markdown

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
database = root / 'data/processed/database.duckdb'
filas_muestra = 10

## Catálogo, tipos de columnas y muestra por tabla

In [ ]:
with duckdb.connect(str(database), read_only=True) as con:
    con.execute('LOAD spatial')
    tablas = con.sql("""SELECT table_name AS tabla FROM information_schema.tables
                        WHERE table_schema = 'silver' ORDER BY table_name""").df()
    display(tablas)
    for tabla in tablas.tabla:
        nombre = 'silver."' + tabla.replace('"', '""') + '"'
        filas = con.execute(f'SELECT count(*) FROM {nombre}').fetchall()[0][0]
        display(Markdown(f'### {tabla} · {filas:,} filas'))
        display(con.sql(f'DESCRIBE {nombre}').df()[['column_name', 'column_type']])
        campos = ('* REPLACE (left(ST_AsText(poligono), 100) AS poligono, ST_AsText(centroide) AS centroide)'
                  if tabla == 'zona' else '*')
        display(con.execute(f'SELECT {campos} FROM {nombre} LIMIT ?', [filas_muestra]).df())

## Consulta editable

In [ ]:
consulta = """SELECT * FROM silver.renta WHERE year(año) = 2024 ORDER BY id_zona LIMIT 20"""
with duckdb.connect(str(database), read_only=True) as con:
    display(con.sql(consulta).df())